In [ ]:
import torch
from transformers import CLIPProcessor, CLIPModel
from torchvision.datasets import Imagenette
import numpy as np
import tqdm
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import accuracy_score
from torch.utils.data import DataLoader

In [ ]:
model = CLIPModel.from_pretrained("openai/clip-vit-base-patch16").to("cuda")
processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch16")

In [ ]:
train_dataset = Imagenette(root = './data', split = 'train', download = True)
val_dataset = Imagenette(root = './data', split = 'val', download = True)

In [ ]:
def collate_fn(batch):
  images = [item[0] for item in batch]
  labels = [item[1] for item in batch]

  return images, labels

batch_size = 128

train_dataloader = DataLoader(train_dataset, batch_size = batch_size, shuffle = True, collate_fn = collate_fn)
val_dataloader = DataLoader(val_dataset, batch_size = batch_size, shuffle = False, collate_fn = collate_fn)

In [ ]:
LinearModel = nn.Linear(512, 10).to("cuda")
LossFunc = nn.CrossEntropyLoss()
optimizer = optim.Adam(LinearModel.parameters(), lr = 0.001)

In [ ]:
for epoch in range(3):
  for (image, labels) in tqdm.tqdm(train_dataloader, desc = (f"Training epoch {epoch + 1}:")):
    with torch.no_grad():
      inputs = processor(images = image, return_tensors = 'pt', padding = True).to("cuda")
      image_features = model.get_image_features(**inputs).pooler_output
      train_image_features = image_features / image_features.norm(dim = 1, keepdim = True)

    labels_tensor = torch.tensor(labels).to("cuda")

    optimizer.zero_grad()
    outputs = LinearModel(train_image_features)
    loss = LossFunc(outputs, labels_tensor)
    loss.backward()
    optimizer.step()


val_pred = []
val_true = []


for (image, labels) in tqdm.tqdm(val_dataloader, desc = 'Validation:'):
  with torch.no_grad():
    inputs = processor(images = image, return_tensors = 'pt', padding = True).to("cuda")
    image_features = model.get_image_features(**inputs).pooler_output
    val_image_features = image_features / image_features.norm(dim = 1, keepdim = True)

  outputs = LinearModel(val_image_features)
  pred = torch.argmax(outputs, dim = 1)

  val_pred.extend(pred.cpu().tolist())
  val_true.extend(labels)

In [ ]:
MyAccuracy = accuracy_score(val_true, val_pred)

print(f"\nAccuracy = {100 * MyAccuracy:.2f}%")

Accuracy = 99.34%



